In [ ]:
#Visualize env adaptation on phyloTree
#author: Sheng-Kai Hsu
#(envPC computation moved to src/S05_envPC_analysis.R, invoked from 06A_spCoordEnvData.sh;
# life-history-paper supplemental analyses moved to the p_lifeHistoryPoaceae repo,
# notebook/06_envirotyping/supplementalLifeHistory.ipynb)
rm(list=ls())
PHYLOGWAS_ROOT <- Sys.getenv("PHYLOGWAS_ROOT", unset = "/workdir/sh2246/p_phyloGWAS")
GIS_DATA_ROOT <- Sys.getenv("GIS_DATA_ROOT", unset = "/workdir/sh2246/p_evolBNI/data/GIS_env_data")

library(ape)
library(pheatmap)
library(magrittr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(plyr)
library(raster)
library(terra)
library(ggtree)
library(ggnewscale)
library(phytools)
library(phangorn)

In [ ]:
#load metadata (assemblyID to species name)
metadata = read.delim(file.path(PHYLOGWAS_ROOT, "data/Poaceae_metadata_filtered_2025.08.28.tsv"),header = T)
metadata$spTaxa = paste(metadata[,1],metadata[,3],sep = ":")
colnames(metadata)[3] = "latest_name"
rownames(metadata) = metadata$assemblyID

In [ ]:
# load tree
spTre = read.tree(file.path(PHYLOGWAS_ROOT, "output/PoaceaeTree_angiosperm353_astral_filtered_20250407.nwk"))
spTre.rooted = root(spTre,"ASM1935983v1",resolve.root = T)
spTre.rooted$edge.length[is.na(spTre.rooted$edge.length)] = 0.1

In [ ]:
ePCs = readRDS(file.path(PHYLOGWAS_ROOT, "output/ePC_20250804.rds"))

commonID = intersect(spTre.rooted$tip.label,rownames(ePCs$environmental.features))
commonID = intersect(commonID,metadata$assemblyID)
eTraits_filtered = as.data.frame(ePCs$environmental.features[commonID,])
spTre.rooted.filtered = keep.tip(spTre.rooted,commonID)

In [ ]:
# envPC pipeline illustration: occurrence points for an example species over a WorldClim raster
# (moved back in from 06D_supplFig_envPCpipeline.R - originally split out due to a suspected
# library(raster)/library(terra) conflict with dplyr/ggplot2; empirically no such conflict
# exists with the packages currently used elsewhere in this notebook, so both are attached here)
coordinates_clean = data.table::fread(file.path(PHYLOGWAS_ROOT, 'output/metadataFormalOut/coordinates_clean.csv'))
coordinates_clean <-
  coordinates_clean %>%
  ddply(.(scientificName),plyr::mutate,envScientificName = paste0('env_',scientificName,'_',1:length(decimalLatitude)))
coordinates_clean <- coordinates_clean %>% na.omit()

wc_bio1 = subset(raster::brick(file.path(GIS_DATA_ROOT, "WorldClim_raw_2.5m_files/wc2.1_2.5m_bio/wc2.1_2.5m_bio_1.tif")),1)
set.seed(1234)
occCoords <- coordinates_clean[grep("Andropogon gerardii",coordinates_clean$scientificName),]
occCoords <- occCoords %>%
  group_by(scientificName) %>%
  sample_n(size = min(500, n()), replace = FALSE)

v <- terra::vect(occCoords, c("decimalLongitude", "decimalLatitude"), crs="+proj=longlat")
vv <- terra::project(v, terra::crs(wc_bio1))

png(file.path(PHYLOGWAS_ROOT, "output/figure/suppFig/FigS1_envPCPipeline_a1.png"),
    width = 6,height = 3,units = "cm",res = 600,pointsize = 6)
par(mar =c (2,2,1,1))
plot(wc_bio1,col = colorRampPalette(c("blue", "lightblue", "yellow","red"))(255),
     ylim = c(-60,70),cex.axis = .75,asp = NA,legend = F,mgp = c(3,.5,0))
points(vv, cex=.1,col=scales::alpha("forestgreen",.5),pch = 1)
dev.off()

In [ ]:
ePC_long <- as.data.frame(ePCs$synthetic.environmental.traits[,1:3]) %>%
  pivot_longer(cols = starts_with("envPC"), names_to = "envPC", values_to = "value")

ePC_long$envPC <- factor(ePC_long$envPC,
                     levels = c("envPC_1","envPC_2","envPC_3"),
                     labels = c("envPC1\ntemperature", "envPC2\nevapotranspiration", "envPC3\nsoil permeability"))

# horizontal histograms

In [ ]:
options(repr.plot.width=5, repr.plot.height=15)

p <- ggplot(ePC_long, aes(value)) +
    geom_histogram(bins = 30, fill = "steelblue", color = "white") +
    facet_wrap(~envPC, ncol = 1, scales = "free",strip.position = "left") +
    coord_flip() + 
    theme_bw() +
    theme(
    strip.placement = "outside",
    strip.background = element_rect(fill = "grey95"),
    strip.text = element_text(face = "bold",size = 8),
    axis.title.y = element_blank()
    )+
    labs(x = "envPC", y = "Count", title = "")

p

In [ ]:
ggsave(file.path(PHYLOGWAS_ROOT, "output/figure/suppFig/FigS1_envPCPipeline_b_v2.png"),p,device = "png",
       width = 8.7*.6,height = 8.7*1.5,unit = "cm",dpi = 600)

In [ ]:
png(file.path(PHYLOGWAS_ROOT, "output/figure/suppFig/FigS1_envPCPipeline_b.png"),width = 15.85,height = 8.7*.5,
    unit = "cm",res = 600, pointsize = 8)
par(mar = c(3,3,1,.5),mfrow = c(1,5))
for (i in 1:5){
    hist(ePCs$synthetic.environmental.traits[,i],main = "")
}
dev.off()

# KG3 analysis

In [ ]:
KG3_perSpecies = read.delim(file.path(PHYLOGWAS_ROOT, "output/KG3_perSpecies_20250804.txt"))
KG3_class = setNames(KG3_perSpecies$KG3_class, KG3_perSpecies$assemblyID)

In [ ]:
KG3_class = KG3_class[commonID]

In [ ]:
KG3_class[!KG3_class%in%seq(1,31,1)] = NA

In [ ]:
KG3_name = c("Af","Am","As","Aw",
             "BWh","BWk","BSh","BSk",
             "Cfa","Cfb","Cfc","Cwa","Cwb","Cwc","Csa","Csb","Csc",
             "Dfa","Dfb","Dfc","Dfd","Dsa","Dsb","Dsc","Dsd","Dwa","Dwb","Dwc","Dwd",
             "ET","EF")

In [ ]:
KG3Col = c(
  "Af" = "#006400", "Am" = "#228B22", "As" = "#FF4500", "Aw" = "#FF8C00",
  "BWh" = "#FFD700", "BWk" = "#DAA520", "BSh" = "#CD853F", "BSk" = "#8B4513",
  "Cfa" = "#32CD32", "Cfb" = "#3CB371", "Cfc" = "#2E8B57", "Cwa" = "#ADFF2F",
  "Cwb" = "#9ACD32", "Cwc" = "#20B2AA", "Csa" = "#00FF7F", "Csb" = "#66CDAA",
  "Csc" = "#008080", "Dfa" = "#4682B4", "Dfb" = "#4169E1", "Dfc" = "#1E90FF",
  "Dfd" = "#00008B", "Dsa" = "#5F9EA0", "Dsb" = "#4682B4", "Dsc" = "#4169E1",
  "Dsd" = "#27408B", "Dwa" = "#708090", "Dwb" = "#4682B4", "Dwc" = "#1E90FF",
  "Dwd" = "#191970", "ET" = "#87CEEB", "EF" = "#ADD8E6"
)

In [ ]:
KGpd = data.frame(envPC = c(ePCs$synthetic.environmental.traits[commonID,1],
                            ePCs$synthetic.environmental.traits[commonID,2],
                            ePCs$synthetic.environmental.traits[commonID,3]
#                             ePCs$synthetic.environmental.traits[commonID,4],
#                             ePCs$synthetic.environmental.traits[commonID,5]
                           ),
                  group = rep(paste0('envPC',1:3),each = nrow(ePCs$synthetic.environmental.traits[commonID,])),
                  KG3Class = rep(KG3_class,3),KG3Name = rep(KG3_name[KG3_class],3),
                  subtribe = rep(metadata[commonID,]$subtribe,3))
KGpd = na.omit(KGpd)

In [ ]:
sort(unique(KGpd$KG3Name))

In [ ]:
options(repr.plot.width=12, repr.plot.height=8)
p = ggplot(KGpd, aes(x = envPC, y = factor(KG3Name,levels = sort(unique(KG3Name))),
                     fill = factor(KG3Name,levels = sort(unique(KG3Name))))) +
    geom_violin(scale = "width", width = .7,orientation = "y",size = .2) +
#     scale_y_discrete(labels = c("Af" = "Tropical-Rainforest (Af)",
#                                 "Am" = "Tropcial-Monsoon (Am)",
#                                 "As" = "Tropical-Savanna dry summer (As)",
#                                 "Aw" = "Tropical-Savanna dry winter (Aw)",
#                                 "BSh" = "Dry-Semiarid steppe-Hot (BSh)",
#                                 "BSk" = "Dry-Semiarid steppe-Cold (BSk)",
#                                 "BWh" = "Dry-Arid desert-Hot (BWh)",
#                                 "BWk" = "Dry-Arid desert-Cold (BWk)",
#                                 "Cfa" = "Temperate-No dry season-Hot summer (Cfa)",
#                                 "Cfb" = "Temperate-No dry season-Warm summer (Cfb)",
#                                 "Csa" = "Temperate-Dry summer-Hot summer (Csa)",
#                                 "Csb" = "Temperate-Dry summer-Warm summer (Csb)",
#                                 "Cwa" = "Temperate-Dry winter-Hot summer (Cwa)",
#                                 "Cwb" = "Temperate-Dry winter-Warm summer (Cwb)",
#                                 "Dfb" = "Continental-No dry season-Warm summer (Dfb)",
#                                 "Dwa" = "Continental-Dry winter-Hot summer (Dwa)",
#                                 "Dwc" = "Continental-Dry winter-Cold summer (Dwc)"))+
    scale_fill_manual(values = KG3Col) +
    geom_boxplot(width = 0.2, fill = "white", alpha = 0.7,outlier.size = 0.1,size = 0.2) +
    facet_grid(group~ .,scales = 'free',switch = 'y') + 
    coord_flip()+
    theme_bw()+
    theme(legend.position = "none",axis.ticks = element_blank(),
          axis.title.y = element_blank(),axis.text.y = element_blank(),
          axis.title.x = element_blank(), axis.text.x = element_text(size = 6,angle = 90,hjust = 1,vjust = .5),
          strip.placement = "outside",  # Move the strip labels outside the plot
          strip.text = element_blank(),  # Rotate text to be horizontal
          panel.spacing = unit(1.9, "lines"),strip.background = element_rect(fill="NA"))
p

In [ ]:
png(file.path(PHYLOGWAS_ROOT, "output/figure/suppFig/FigS1_envPCPipeline_d_v2.png"),
    width = 8.7*.5,height = 11.93, units = "cm",res = 600,pointsize = 8,bg = "transparent")
p
dev.off()

In [ ]:
names(which(table(KGpd$subtribe)>30))

In [ ]:
sort(table(KGpd$subtribe)/3)

In [ ]:
KGpd_subset = KGpd[KGpd$subtribe%in%names(which(table(KGpd$subtribe)>=30)),]
KGpd_subset$subtribe = factor(KGpd_subset$subtribe,
                              levels = rev(c('Andropogoninae','Anthistiriinae','Saccharinae','Sorghinae',
                                       'Ischaeminae','Apludinae','Chrysopogoninae',
                                       'Tripsacinae','Rhytachninae',
                                       'Ratzeburgiinae','Rottboelliinae',
                                       'Arthraxoninae','Cenchrinae','Eleusininae',
                                       'Triticinae','Hordeinae','Aveninae',"Arundinariinae",'Oryzinae')))

In [ ]:
options(repr.plot.width=12, repr.plot.height=8)
p = ggplot(KGpd_subset, aes(x = envPC, y = subtribe,fill = subtribe)) +
    geom_violin(scale = "width", width = .7,orientation = "y",size = .2) +
    geom_boxplot(width = 0.2, fill = "white", alpha = 0.7,outlier.size = 0.1,size = 0.2) +
    coord_flip()+
    facet_grid(group~.,scales = 'free') + 
    theme_bw()+
    theme(legend.position = "none",axis.ticks.x = element_blank(),axis.ticks.y.left = element_blank(),
          axis.title.y = element_blank(),axis.text.x = element_text(size = 6,angle = 90,vjust =.5,hjust = 1),
          axis.title.x = element_blank(), axis.text.y = element_blank(),
          strip.placement = "outside",  # Move the strip labels outside the plot
          strip.text.y = element_blank(),  # Rotate text to be horizontal
          panel.spacing = unit(1.9, "lines"),strip.background = element_rect(fill="NA"))
p

In [ ]:
png(file.path(PHYLOGWAS_ROOT, "output/figure/suppFig/FigS1_envPCPipeline_e_v2.png"),
    width = 8.7*.5,height = 8.7*1.5,
    units = "cm",res = 600,pointsize = 8,bg = "transparent")
p
dev.off()

In [ ]:
p = ggplot(KGpd[KGpd$group%in%c("envPC1","envPC2"),], aes(x = envPC, y = factor(KG3Name,levels = rev(sort(unique(KG3Name)))),fill = factor(KG3Name,levels = rev(sort(unique(KG3Name)))))) +
    geom_violin(scale = "width", width = .7,orientation = "y",size = 0.2) +
    scale_fill_manual(values = KG3Col) +
    geom_boxplot(width = 0.2, fill = "white", alpha = 0.7,outlier.size = 0.1,size = 0.2) +
    facet_wrap(~group,scales = 'free_x') + 
    theme_bw()+
    theme(legend.position = "none",
          plot.background = element_rect(fill = "transparent", colour = NA),
          panel.background = element_rect(fill = "transparent", colour = NA),
          axis.title.y = element_blank(),axis.text.y = element_text(face = 'bold',size =8,colour = KG3Col[rev(sort(unique(KGpd$KG3Name)))]),
          axis.title.x = element_blank(), #axis.text.x = element_blank(),
          strip.placement = "outside",  # Move the strip labels outside the plot
          strip.text.x = element_text(angle = 0,size = 12),  # Rotate text to be horizontal
          panel.spacing = unit(1, "lines"),strip.background = element_rect(fill="transparent"))
p
png(file.path(PHYLOGWAS_ROOT, "output/figure/Fig1b.png"),width = 8.7*.6,height = 8.7, units = "cm",res = 900,pointsize = 8,bg = "transparent")
p
dev.off()

# tree overlaying

In [ ]:
tribeLab = metadata[commonID,]$tribe
names(tribeLab) = commonID
# tribeLab = na.omit(tribeLab)

In [ ]:
tmpTrait = ePCs$synthetic.environmental.traits[commonID,1]
tmpTrait2 = ePCs$synthetic.environmental.traits[commonID,2]
tmpTrait3 = ePCs$synthetic.environmental.traits[commonID,3]
names(tmpTrait)=commonID
names(tmpTrait2)=commonID
names(tmpTrait3)=commonID

tmpTrait4 = KG3_class[commonID]
tmpTrait4 = KG3_name[tmpTrait4]
names(tmpTrait4)=commonID
# fit <- phytools::fastAnc(spTre.rooted.filtered,tmpTrait , vars=TRUE, CI=TRUE)
# td <- data.frame(node = nodeid(spTre.rooted.filtered, commonID),
#                envPC1 = tmpTrait)
# nd <- data.frame(node = names(fit$ace), envPC1 = fit$ace)
# d <- rbind(td, nd)
# d$node <- as.numeric(d$node)
# plotTree <- full_join(spTre.rooted.filtered, d, by = 'node')

# fit2 <- phytools::fastAnc(spTre.rooted.filtered,tmpTrait2 , vars=TRUE, CI=TRUE)
# td2 <- data.frame(node = nodeid(spTre.rooted.filtered, commonID),
#                envPC2 = tmpTrait2)
# nd2 <- data.frame(node = names(fit2$ace), envPC2 = fit2$ace)
# d2 <- rbind(td2, nd2)
# d2$node <- as.numeric(d2$node)
# plotTree2 <- full_join(spTre.rooted.filtered, d2, by = 'node')

In [ ]:
tmpTrait5 = metadata[commonID,]$subtribe
names(tmpTrait5)=commonID

In [ ]:
head(tmpTrait5)

In [ ]:
highlightNode = c()
for( i in c("Andropogoneae","Cynodonteae","Oryzeae","Paniceae","Poeae","Triticeae")){
    highlightNode = c(highlightNode,getMRCA(spTre.rooted.filtered,na.omit(metadata[commonID,][tribeLab==i,1])))
}


highlightNodeDat= data.frame(node = highlightNode,
                             Tribe = c("Andropogoneae","Cynodonteae","Oryzeae","Paniceae","Poeae","Triticeae"))

In [ ]:
highlightNode2 = c(562,534,480,411,442,388,370,324,277,309,247,239,265,225,200,160,13,39,41,54,98,118)+length(spTre.rooted.filtered$tip.label)
highlightNodeDat2= data.frame(node = highlightNode2,
                             Tribe = c('Andropogoninae','Anthistiriinae','Anthistiriinae','Saccharinae','Sorghinae',
                                       'Ischaeminae','Apludinae','Chrysopogoninae',
                                       'Tripsacinae','Rhytachninae',
                                       'Ratzeburgiinae','Rottboelliinae','Rottboelliinae',
                                       'Arthraxoninae','Cenchrinae','Eleusininae',
                                       'Triticinae','Hordeinae','Hordeinae','Aveninae','Arundinariinae','Oryzinae'))

In [ ]:
highlightNode2

In [ ]:
spTre.rooted.filtered.nodeLabeled = makeNodeLabel(spTre.rooted.filtered)

In [ ]:
subtribeCol = c(RColorBrewer::brewer.pal(8,"Set1"),RColorBrewer::brewer.pal(11,"Set3"))

In [ ]:
p <- ggtree(spTre.rooted.filtered,layout = "rectangular", ladderize = T,size = .3)+
    geom_cladelab(node=highlightNodeDat2$node, align=T,angle = 60,fontsize = 0,offset = 0,hjust = 0.3,vjust =2,
                 barcolor = subtribeCol[as.factor(highlightNodeDat2$Tribe)],barsize = 2,
                  label = as.factor(highlightNodeDat2$Tribe))

p1 <- p+ new_scale_fill()
p1 <- gheatmap(p1, as.data.frame(tmpTrait4) , offset=1.7, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_manual(values = KG3Col,name = "KG3 classification")
p1 <- p1+ new_scale_fill()
p1 <- gheatmap(p1, as.data.frame(tmpTrait) , offset=4.0, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_gradientn(colours = c('blue',"lightblue",'red'),name = "envPC1")
p2 <- p1+ new_scale_fill()
p2 <- gheatmap(p2, as.data.frame(tmpTrait2) , offset=6.3, width=0.1,colnames = F,color = NA) +
    scale_x_ggtree() + 
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_gradientn(colours = c('#1B5E20',"yellowgreen",'goldenrod'),name = "envPC2")
p3 <- p2+ new_scale_fill()
p3 <- gheatmap(p3, as.data.frame(tmpTrait3) , offset=8.6, width=0.1,colnames = F,color = NA) +
    scale_y_continuous(expand=c(0, 0.3))+
    scale_fill_gradientn(colours = c('#4E342E','tan',"#D7CCC8"),name = "envPC3")+
    theme(axis.text=element_text(size=5),legend.position = "right",legend.key.width = unit(3,units = "mm"),
          legend.key.height = unit(3,units = "mm"),
          legend.margin = margin(t = -2,r =0,b = 0,l =0, unit='mm'),
          plot.margin = margin(t= 1, r= 1,b = 1,l = 1, unit='mm'),
          legend.title = element_text(size = 8),legend.text = element_text(size = 6))
p3
png(file.path(PHYLOGWAS_ROOT, "output/figure/Fig1a_v2.png"),width = 8.7*1.5,height = 17.4,units = "cm",
    pointsize = 6,res = 600)
p3
dev.off()

In [ ]:
envpd = data.frame(envPC = c(tmpTrait,tmpTrait2,tmpTrait3),taxagroup = rep(metadata[commonID,]$subtribe,3),
                   group = rep(c('envPC1','envPC2','envPC3'),each = length(commonID)))
envpd_subset = envpd[envpd$taxagroup%in%names(which(table(envpd$taxagroup)>=30)),] #subtribe with >=10 individual

In [ ]:
envpd_subset$taxagroup = factor(envpd_subset$taxagroup,
                                levels = c('Andropogoninae','Anthistiriinae','Saccharinae','Sorghinae',
                                       'Ischaeminae','Apludinae','Chrysopogoninae',
                                       'Tripsacinae','Rhytachninae',
                                       'Ratzeburgiinae','Rottboelliinae',
                                       'Arthraxoninae','Cenchrinae','Eleusininae',
                                       'Triticinae','Hordeinae','Aveninae','Arundinariinae','Oryzinae'))

In [ ]:
options(repr.plot.width=12, repr.plot.height=8)
par(mar=c(15,3,2,2),mfrow = c(1,2),las =2)
KG3Count = t(table(tmpTrait4,tribeLab))
KG3Freq = KG3Count/rowSums(KG3Count)
bp = barplot(t(KG3Freq[rowSums(KG3Count)>=30,]),col = KG3Col[colnames(KG3Freq[rowSums(KG3Count)>=30,])],ylim = c(0,1.2),
             cex.names = 2,cex.axis =1.5,args.legend = list(x= "top",bty= 'n',horiz =T,cex = 1.5),yaxt = 'n',las = 2)
axis(2,seq(0,1,0.2))

KG3Count = t(table(tmpTrait4,metadata[commonID,]$subtribe))
KG3Freq = KG3Count/rowSums(KG3Count)
bp = barplot(t(KG3Freq[levels(envpd_subset$taxagroup),]),col = KG3Col[colnames(KG3Freq)],ylim = c(0,1.2),
             cex.names = 2,cex.axis =1.5,args.legend = list(x= "top",bty= 'n',horiz =T,cex = 1.5),yaxt = 'n',las = 2)
axis(2,seq(0,1,0.2))


In [ ]:
# options(repr.plot.width=8, repr.plot.height=8)
png(file.path(PHYLOGWAS_ROOT, "output/figure/Fig1c_v2.png"),width = 8.7*.75,height = 8.7,units = "cm",pointsize = 8,res = 900)
par(mar=c(3,6,2,2),las =2)
barplot(t(KG3Freq[rev(levels(envpd_subset$taxagroup)),]),col = KG3Col[colnames(KG3Freq)],horiz = T,
        cex.names = .75,cex.axis =.75,args.legend = list(x= "top",bty= 'n',horiz =T,cex = 1.5),las = 1)
# axis(2,seq(0,1,0.2),cex.axis = 1)
dev.off()

In [ ]:
options(repr.plot.width=10, repr.plot.height=6)
p = ggplot(envpd_subset, aes(x = taxagroup, y = envPC,fill = taxagroup)) +
    geom_violin(scale = "width", width = .7) +
    geom_boxplot(width = 0.2, fill = "white", alpha = 0.7) +
    facet_wrap(~group,nrow = 2,switch = 'y',scales = 'free_y') + 
    theme_bw()+
    theme(legend.position = "none",
          axis.title.y = element_blank(),axis.text.y = element_text(size = 10),
          axis.text.x = element_text(angle = 90, size = 10),axis.title.x = element_blank(),
          strip.placement = "outside",  # Move the strip labels outside the plot
          strip.text.y = element_text(angle = 0,size = 12),  # Rotate text to be horizontal
          panel.spacing = unit(1, "lines"))
p

In [ ]:
options(repr.plot.width=8, repr.plot.height=8)
par(mar=c(3,6,2,2),las =2)
barplot(t(KG3Freq[rev(levels(envpd_subset$taxagroup)),]),col = KG3Col[colnames(KG3Freq)],horiz = T,
        cex.names = .75,cex.axis =.75,args.legend = list(x= "top",bty= 'n',horiz =T,cex = 1.5),las = 1)


In [ ]:
subtribeTopology = '((((((((((Andropogoninae,Anthistiriinae),(Saccharinae,Sorghinae)),Ischaeminae),
Apludinae),Chrysopogoninae),((Rhytachninae,Tripsacinae),(Rottboelliinae,Ratzeburgiinae))),Arthraxoninae),
Cenchrinae),Eleusininae),((((Triticinae,Hordeinae),Aveninae),Arundinariinae),Oryzinae));'

In [ ]:
png(file.path(PHYLOGWAS_ROOT, "output/figure/Fig1c_tree_v2.png"),width = 8.7*.25,height = 8.7,units = "cm",pointsize = 8,res = 900)
ggtree(read.tree(text = subtribeTopology),ladderize = F)+
    geom_tippoint(fill = subtribeCol[as.factor(unique(highlightNodeDat2$Tribe))],pch = 21)+
    scale_y_reverse()
dev.off()

# ancestral state reconstruction & transition identification

In [ ]:
library(ape)
library(phangorn)

# Keep only the most ancestral nodes among a set (nodes can include tips)
keep_most_ancestral <- function(tree, nodes) {
  stopifnot(inherits(tree, "phylo"))
  nodes <- unique(as.integer(nodes))

  max_id <- Ntip(tree) + tree$Nnode
  nodes <- nodes[nodes >= 1 & nodes <= max_id]
  if (length(nodes) <= 1) return(nodes)

  # Depth from root (smaller = more ancestral). Works for tips + internal nodes.
  depth <- node.depth.edgelength(tree)
  depth <- depth[nodes]
  names(depth) <- as.character(nodes)

  # Tip sets for each node (tips return themselves)
  tipset <- lapply(nodes, function(nd) sort(Descendants(tree, nd, type = "tips")[[1]]))
  names(tipset) <- as.character(nodes)

  # Helper: is A subset of B?
  is_subset <- function(A, B) {
    length(A) <= length(B) && all(A %in% B)
  }

  redundant <- logical(length(nodes))

  for (i in seq_along(nodes)) {
    Ai <- tipset[[i]]
    di <- depth[i]

    for (j in seq_along(nodes)) {
      if (i == j) next
      Bj <- tipset[[j]]
      dj <- depth[j]

      # If i's clade is contained in j's clade, i is redundant unless:
      # - they are identical and i is more ancestral (smaller depth)
      if (is_subset(Ai, Bj)) {
        identical_sets <- length(Ai) == length(Bj)
        if (!identical_sets) {
          redundant[i] <- TRUE
          break
        } else {
          # identical tip-set: keep the more ancestral one (smaller depth)
          if (di > dj) {
            redundant[i] <- TRUE
            break
          }
        }
      }
    }
  }

  nodes[!redundant]
}

# Example:
# cleaned <- keep_most_ancestral(tr, nodes_of_interest)


In [ ]:
# Subset tree to include only species with trait data
rooted_tree <- spTre.rooted.filtered
# rooted_tree$edge.length[rooted_tree$edge.length==0.01] = 0.1 

# Step 3: Subset to match trait data
tree.sub <- keep.tip(rooted_tree, rownames(ePCs$synthetic.environmental.traits))
# Step 2: Make the tree fully dichotomous (resolve polytomies randomly)
tree.sub <- multi2di(tree.sub)

transitionNodes = list()
for (i in 1:3){
    # Extract trait data and ensure it's in the right order
    trait <- na.omit(ePCs$synthetic.environmental.traits[,i])
    names(trait) = rownames(ePCs$synthetic.environmental.traits)
    trait <- trait[tree.sub$tip.label]  # match the order of the tips

    # Use ace() for discrete traits (method = "ML")
    fit <- fastAnc(tree.sub,trait, var =T, CI = T)

    # Create data frame for tips
    td <- data.frame(node = 1:Ntip(tree.sub),ace = trait, var = 0,upper = trait,lower=trait)

    # Create data frame for internal nodes
    nd <- data.frame(node = (Ntip(tree.sub) + 1):(Ntip(tree.sub) + Nnode(tree.sub)),
                     ace = fit$ace,var = fit$var,upper = fit$CI95[,2],lower = fit$CI95[,1])

    # Combine tip and node data
    d <- rbind(td, nd)
    d$node <- as.numeric(d$node)
    plotTree <- full_join(tree.sub, d, by = 'node')

    edgeState = apply(plotTree@phylo$edge,c(1,2),function(x) d[d$node==x,2])
    edgeStateUpper = apply(plotTree@phylo$edge,c(1,2),function(x) d[d$node==x,4])
    edgeStateLower = apply(plotTree@phylo$edge,c(1,2),function(x) d[d$node==x,5])

    # toC = edgeStateLower[,1]>edgeStateUpper[,2]&edgeState[,2]<edgeState[,1]
    # toW = edgeStateLower[,2]>edgeStateUpper[,1]&edgeState[,2]>edgeState[,1]

    toL = edgeStateLower[,1]>edgeStateUpper[,2]&edgeState[,2]<quantile(trait,.3)&edgeState[,1]>quantile(trait,.3)
    toH = edgeStateLower[,2]>edgeStateUpper[,1]&edgeState[,2]>quantile(trait,.7)&edgeState[,1]<quantile(trait,.7)
    sum(toL,na.rm = T)
    sum(toH,na.rm = T)

    nodetoH = keep_most_ancestral(plotTree@phylo,plotTree@phylo$edge[toH,2])
    nodetoL = keep_most_ancestral(plotTree@phylo,plotTree@phylo$edge[toL,2])
    transitionNodes[[i]] = list(nodetoH,nodetoL)
}


In [ ]:
write.tree(tree.sub,file.path(PHYLOGWAS_ROOT, "output/powerSimulation/tree.nwk"))

In [ ]:
unlist(lapply(transitionNodes,function(x) sapply(x,length)[c(2,1)]))

In [ ]:
write(rjson::toJSON(transitionNodes),file.path(PHYLOGWAS_ROOT, "output/powerSimulation/realTransition.json"))

In [ ]:
tree.sub$node.label = round(as.numeric(tree.sub$node.label),2)

In [ ]:
tree.sub$node.label[1] = 1 #root node

In [ ]:
p <- ggtree(tree.sub,layout = "rectangular", ladderize = T,size = .3)

In [ ]:
merge_tab5=data.frame(assemblyID=p$data$label)
merge_tab5=merge(merge_tab5,metadata,by = "assemblyID")
merge_tab5=merge_tab5[!duplicated(merge_tab5$assemblyID),]
rownames(merge_tab5)=merge_tab5$assemblyID
merge_tab5=merge_tab5[p$data$label,]

p$data$tip.label = NA
p$data$tip.label[!is.na(merge_tab5$spTaxa)] = merge_tab5$spTaxa[!is.na(merge_tab5$spTaxa)]

In [ ]:
tmp1 = cut(tmpTrait,breaks = quantile(tmpTrait,c(0,0.3,0.7,1)),include.lowest = T,labels = c("cold","unclassified","warm"))
names(tmp1) = names(tmpTrait)

In [ ]:
p1 <- p +
    geom_point(color=ifelse(p$data$node%in%transitionNodes[[1]][[1]],"red",NA), alpha=1, size=1.5,shape = 18) +  
    geom_point(color=ifelse(p$data$node%in%transitionNodes[[1]][[2]],"blue",NA), alpha=1, size=1.5,shape = 18)    
p2 <- p + 
    geom_point(color=ifelse(p$data$node%in%transitionNodes[[2]][[1]],"goldenrod",NA), alpha=1, size=1.5,shape = 18) +  
    geom_point(color=ifelse(p$data$node%in%transitionNodes[[2]][[2]],"#1B5E20",NA), alpha=1, size=1.5,shape = 18)
p3 <- p +
    geom_point(color=ifelse(p$data$node%in%transitionNodes[[3]][[1]],"#D7CCC8",NA), alpha=1, size=1.5,shape = 18) +  
    geom_point(color=ifelse(p$data$node%in%transitionNodes[[3]][[2]],"#4E342E",NA), alpha=1, size=1.5,shape = 18)

p1+p2+p3

In [ ]:
png(file.path(PHYLOGWAS_ROOT, "output/figure/envPC_ASR_transitionNodes.png"),width = 48,height = 48,units = "cm",
    pointsize = 6,res = 600)
p1+p2+p3
dev.off()


In [ ]:
d <- p$data

node_coords = c()
for (i in 1:3){
    for (j in 1:2){
        node_coords <- rbind(node_coords,d[d$node %in% transitionNodes[[i]][[j]],])
    }
}


In [ ]:
unlist(lapply(transitionNodes,function(x) sapply(x,length)))

In [ ]:
annotations <- data.frame(
  node  = unlist(lapply(transitionNodes,unlist)),
  label = c(rep("envPC1_high\nhigh temperature",27),rep("envPC1_low\ncold & short season",19),
            rep("envPC2_high\nwater deficit",21),rep("envPC2_low\nrainy & humid",22),
            rep("envPC3_high\npoor drainage",27),rep("envPC3_low\nwater/nutrient leaching",30)),
  color = c(rep("red",27),rep("blue",19),
            rep("goldenrod",21),rep('#1B5E20',22),
            rep("#D7CCC8",27),rep('#4E342E',30))
)


# Merge with tree coordinates
anno <- merge(annotations, d[, c("node","x","y","parent")], by="node")

parent_coords <- d[, c("node","x")]
names(parent_coords) <- c("parent","x_parent")
anno <- merge(anno, parent_coords, by="parent")

# Midpoint x on the branch
anno$x_mid <- (anno$x + anno$x_parent) / 2

In [ ]:
# Arrow length and label offset (tune to your tree)
arrow_len <- 0.8    # vertical length of arrow
label_off <- 0.2    # gap between arrow tail and label

library(grid)

# Custom key that draws a downward arrow
draw_key_down_arrow <- function(data, params, size) {
  segmentsGrob(
    x0 = 0.5, x1 = 0.5,        # vertical: same x
    y0 = 0.6, y1 = 0.4,        # top to bottom
    gp = gpar(
      col = data$colour,fill = data$colour,
      lwd = data$linewidth * .pt
    ),
    arrow = arrow(length = unit(0.2, "cm"), type = "closed", ends = "last")
  )
}

# Assign the custom key to geom_segment
p +
  xlim(c(0, 25)) +
  geom_tiplab(aes(label = tip.label), hjust = -.025, size = 1) +
  geom_segment(
    data = anno,
    aes(x     = x_mid,
        xend  = x_mid,
        y     = y + arrow_len,
        yend  = y,
        color = label),
    arrow = arrow(length = unit(0.2, "cm"), type = "closed", ends = "last"),
    linewidth = 0.1,
    show.legend = TRUE,
    key_glyph = draw_key_down_arrow    # ← plug in custom key here
  ) +
  scale_color_manual(
    name   = "Class",
    values = setNames(anno$color, anno$label)
  ) +
  theme(legend.position = "right")

In [ ]:
# png(file.path(PHYLOGWAS_ROOT, "output/figure/FigS6.png"),
#     height = 60,width = 60, unit = "cm",pointsize = 6,res = 600)
pdf(file.path(PHYLOGWAS_ROOT, "output/figure/FigS6_v2.pdf"),
    height = 30,width = 20,pointsize = 6)
p +
  geom_nodelab(hjust = 0,nudge_x = -0.05,color = "darkred",size = 1)+
  xlim(c(0, 20)) +
  geom_tiplab(aes(label = tip.label), hjust = -.025, size = 1) +
  geom_segment(
    data = anno,
    aes(x     = x_mid,
        xend  = x_mid,
        y     = y + arrow_len,
        yend  = y,
        color = label),
    arrow = arrow(length = unit(0.2, "cm"), type = "closed", ends = "last"),
    linewidth = 0.1,
    show.legend = TRUE,
    key_glyph = draw_key_down_arrow    # ← plug in custom key here
  ) +
  scale_color_manual(
    name   = "Class",
    values = setNames(anno$color, anno$label)
  ) +
  theme(legend.position = "right")
dev.off()

In [ ]:
transitionByTribe = c()
for (i in 1:nrow(highlightNodeDat2)){
    allDesAfterTransition = phangorn::Descendants(tree.sub,node = highlightNodeDat2[i,1],"all")
    tmp = c()
    for (j in 1:3){
        tmp1 = table(transitionNodes[[j]][[1]]%in%allDesAfterTransition)
        tmp2 = table(transitionNodes[[j]][[2]]%in%allDesAfterTransition)
        tmp = c(tmp,c(tmp2[2],tmp1[2]))
    }
    transitionByTribe = rbind(transitionByTribe,tmp)   
}
rownames(transitionByTribe) = highlightNodeDat2$Tribe

In [ ]:
transitionByTribe[is.na(transitionByTribe)]=0
transitionByTribe = apply(transitionByTribe,2,function(x) tapply(x,rownames(transitionByTribe),sum)[levels(envpd_subset$taxagroup)])

In [ ]:
pheatmap(transitionByTribe,cluster_rows = F,cluster_cols = F,show_rownames = F,show_colnames = F,legend = F,
         display_numbers = T,number_format = "%0d",number_color = ifelse(transitionByTribe>5,"white","black"),
         color = RColorBrewer::brewer.pal(9,'RdPu'),fontsize = 24,
         filename = file.path(PHYLOGWAS_ROOT, "output/figure/Fig1d_v2.png"),
         width = 7.4,height = 7.4,units = "cm",pointsize = 12,res = 900)
